In [ ]:
#Google Colab setup for unsloth

%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#Lightning.ai setup for unsloth

%pip uninstall -y unsloth unsloth_zoo trl transformers
%pip install --no-cache-dir "transformers==4.56.2" "trl==0.22.2"
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [2]:
# For Windows locally
import os, sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DATASETS_MULTITHREADING_MAX_WORKERS"] = "1"

if sys.platform == "win32":
    from datasets import Dataset

    if not hasattr(Dataset, "_original_map_windows_fix"):
        Dataset._original_map_windows_fix = Dataset.map

        def _win32_safe_map(self, *args, **kwargs):
            kwargs["num_proc"] = None   # force single-process map
            return Dataset._original_map_windows_fix(self, *args, **kwargs)

        Dataset.map = _win32_safe_map

In [3]:
from unsloth import FastLanguageModel
import torch

import os
HF_USERNAME = os.getenv('HF_USERNAME', 'RealPirate786')
HF_TOKEN = os.getenv('HF_TOKEN')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0704 20:16:40.949000 27196 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length = 2048
lora_rank = 32
model_path = r"D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.65, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.12.1. vLLM: 0.24.0+cu132.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The tokenizer you are loading from 'D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading th

#### Define System Prompt

In [ ]:
# @title
def get_system_prompt():
    
  system_prompt = '''You are playing Minesweeper.

Rules:
- Board uses 0-indexed row and column.
- _ means unknown.
- F means flagged.
- Numbers 0-8 are revealed cells.
- Choose exactly one action.
- Output only valid JSON.

Allowed actions:
{"action":"reveal","x":int,"y":int}
{"action":"flag","x":int,"y":int}

Board:
row 0: _ _ _ _ _
row 1: _ 1 1 _ _
row 2: _ 1 F _ _
row 3: _ _ _ _ _
row 4: _ _ _ _ _


Return one action as JSON only.
  '''
  return system_prompt


system_prompt = get_system_prompt()

#### Define Custom Chat Template

In [6]:
def get_chat_template():
    chat_template = '''
    {%- set eos = eos_token if eos_token is defined and eos_token is string and eos_token != '<|im_end|>' else '' -%}

    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}
    {%- endif %}

    {%- for message in messages %}
        {%- if message.content is string %}
            {%- set content = message.content %}
        {%- else %}
            {%- set content = '' %}
        {%- endif %}

        {%- if message.role == "user" %}
            {{- '<|im_start|>user\\n' + content + '<|im_end|>\\n' }}

        {%- elif message.role == "system" and not loop.first %}
            {{- '<|im_start|>system\\n' + content + '<|im_end|>\\n' }}

        {%- elif message.role == "assistant" %}
            {%- set reasoning_content = '' %}

            {%- if message.reasoning_content is string %}
                {%- set reasoning_content = message.reasoning_content %}
            {%- else %}
                {%- if '</think>' in content %}
                    {%- set reasoning_content = content.split('</think>')[0].rstrip('\\n').split('<think>')[-1].lstrip('\\n') %}
                    {%- set content = content.split('</think>')[-1].lstrip('\\n') %}
                {%- endif %}
            {%- endif %}

            {%- if reasoning_content %}
                {{- '<|im_start|>assistant\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') + '<|im_end|>' + eos + '\\n' }}
            {%- else %}
                {{- '<|im_start|>assistant\\n' + content + '<|im_end|>' + eos + '\\n' }}
            {%- endif %}
        {%- endif %}
    {%- endfor %}

    {%- if add_generation_prompt %}
        {{- '<|im_start|>assistant\\n' }}
    {%- endif %}
    '''
    return chat_template

chat_template = get_chat_template()

In [7]:
tokenizer.chat_template = chat_template

#### Load The Datasets for SFT

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
dataset = pd.read_csv(r"D:\Projects\Minesweeper LLM\dataset\minesweeper_5x5.csv")
train, test = train_test_split(dataset, test_size = 0.3, random_state=42, shuffle=True)

#### Format the dataset

In [11]:
def format_dataset_train(df):
    input = f"Board State: {df['input']}\nMax Mines: {df['max_mines']}\nMax Rows: {df['max_rows']}\nMax Columns: {df['max_columns']}"
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
        {"role": "assistant", "content": output},
    ]

def format_dataset_test(df):
    input = f"Board State: {df['input']}\nMax Mines: {df['max_mines']}\nMax Rows: {df['max_rows']}\nMax Columns: {df['max_columns']}"
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
    ]

train['Messages'] = train.apply(format_dataset_train, axis=1)
test['Messages'] = test.apply(format_dataset_test, axis=1)

#check the formatted messages
test['Messages'][0]

[{'role': 'system',
  'content': 'You are playing Minesweeper.\n\nRules:\n- Board uses 0-indexed row and column.\n- _ means unknown.\n- F means flagged.\n- Numbers 0-8 are revealed cells.\n- Choose exactly one action.\n- Output only valid JSON.\n\nAllowed actions:\n{"action":"reveal","row":int,"col":int}\n{"action":"flag","row":int,"col":int}\n\nBoard:\nrow 0: _ _ _ _ _\nrow 1: _ 1 1 _ _\nrow 2: _ 1 F _ _\nrow 3: _ _ _ _ _\nrow 4: _ _ _ _ _\n\n\nReturn one action as JSON only.\n  '},
 {'role': 'user',
  'content': 'Board State: [["_", "_", "1", ".", "."], ["_", "_", "1", "1", "1"], ["_", "_", "_", "_", "_"], ["_", "1", "1", "2", "1"], ["_", "1", ".", ".", "."]]\nMax Mines: 3\nMax Rows: 5\nMax Columns: 5'}]

#### Convert to HuggingFace compatible dataset

In [12]:
from datasets import Dataset

train["text"] = tokenizer.apply_chat_template(train["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)
test["text"] = tokenizer.apply_chat_template(test["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)

train = Dataset.from_pandas(train)
test = Dataset.from_pandas(test)

#### Start the Finetuning-Process

In [13]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc =1,
        dataloader_num_workers=0,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 7e-6, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/2606 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [14]:
batch = next(iter(trainer.get_train_dataloader()))

input_ids = batch["input_ids"][0].cpu()
labels = batch["labels"][0].cpu()

for i in range(len(input_ids) - 80, len(input_ids)):
    tid = int(input_ids[i])
    lid = int(labels[i])

    token = tokenizer.decode([tid], skip_special_tokens=False)

    if lid == -100:
        print(i, "MASKED ", repr(token))
    else:
        print(i, "TRAINED", repr(token))

159 TRAINED '".'
160 TRAINED '",'
161 TRAINED ' ".",'
162 TRAINED ' ".",'
163 TRAINED ' ".",'
164 TRAINED ' "."'
165 TRAINED '],'
166 TRAINED ' ['
167 TRAINED '".'
168 TRAINED '",'
169 TRAINED ' ".",'
170 TRAINED ' "'
171 TRAINED '1'
172 TRAINED '",'
173 TRAINED ' "'
174 TRAINED '1'
175 TRAINED '",'
176 TRAINED ' "."'
177 TRAINED '],'
178 TRAINED ' ['
179 TRAINED '".'
180 TRAINED '",'
181 TRAINED ' ".",'
182 TRAINED ' "'
183 TRAINED '2'
184 TRAINED '",'
185 TRAINED ' ".",'
186 TRAINED ' "."'
187 TRAINED '],'
188 TRAINED ' ['
189 TRAINED '".'
190 TRAINED '",'
191 TRAINED ' ".",'
192 TRAINED ' ".",'
193 TRAINED ' ".",'
194 TRAINED ' "."'
195 TRAINED ']]\n'
196 TRAINED 'Max'
197 TRAINED ' Mines'
198 TRAINED ':'
199 TRAINED ' '
200 TRAINED '3'
201 TRAINED '\n'
202 TRAINED 'Max'
203 TRAINED ' Rows'
204 TRAINED ':'
205 TRAINED ' '
206 TRAINED '5'
207 TRAINED '\n'
208 TRAINED 'Max'
209 TRAINED ' Columns'
210 TRAINED ':'
211 TRAINED ' '
212 TRAINED '5'
213 TRAINED '<|im_end|>'
214 TRAINED '\n'

Run the trainer

In [ ]:
try:
    trainer.train()
except KeyboardInterrupt:
    print("Training interrupted.")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,606 | Num Epochs = 2 | Total steps = 5,212
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kartikeya Srivastava\_netrc.
wandb: Currently logged in as: kartikeyasrivastava769 (kartikeyasrivastava769-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,4.307394
10,3.626138
15,2.766576
20,2.022855
25,1.554542
30,1.178563
35,0.809474
40,0.618559
45,0.474868
50,0.374895


Training interrupted.


wandb: WARNING Tried to log to step 1 that is less than the current step 145. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 165. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 185. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 205. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 5 that is less than the current step 225. Steps must be monotonically increasing, so this data will be ignored. See https://wand

#### Test the trained model

In [17]:
from transformers import TextStreamer
FastLanguageModel.for_inference(model)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

stop_ids = [tokenizer.eos_token_id]

if (
    im_end_id is not None
    and im_end_id != tokenizer.unk_token_id
    and im_end_id not in stop_ids
):
    stop_ids.append(im_end_id)

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)
text = tokenizer.apply_chat_template(
    test["Messages"][56],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens=32,
    do_sample=False,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    streamer=streamer,
)
print('---')

Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 {"action": "reveal", "x": 3, "y": 4}
---


#### Save the model

Save as .safetensors 16-bit

In [18]:
# Save the model to Hugging Face Hub. You can load this model later for further training using GRPO or for inference.
def save_model():
    model.save_pretrained_merged("Minesweeper_agent_Qwen3_0.6B", tokenizer, save_method = "merged_16bit",)
    model.push_to_hub_merged(f"{HF_USERNAME}/Minesweeper_agent_Qwen3_0.6B-SFT", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)

# save_model() #Uncomment this to save model

Save as GGUF format

In [19]:
# Save the model in GGUF format for using it through llama.cpp
def save_model_gguf():
    model.save_pretrained_gguf("Minesweeper_agent_Qwen3_4B_2507", tokenizer,)
    model.push_to_hub_gguf(f"{HF_USERNAME}/Minesweeper_agent_Qwen3_4B_2507-GGUF", tokenizer, token = HF_TOKEN, quantization_method = 'q8_0')

# save_model_gguf() #Uncomment this to save model

### Reinforcement Learning with GRPO

In [ ]:
#Use this if you want to use your own pre-trained model. Useful when you save previous model first first and then want to load it for further training using GRPO. No need to use this if you are training model directly after SFT.
from unsloth import FastLanguageModel
max_seq_length = 2048
lora_rank = 32
model_path = r"D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B"
if False: 
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = max_seq_length,
        load_in_4bit = True, # False for LoRA 16bit
        fast_inference = False, # Enable vllm fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.75, # Reduce if out of memory
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank * 2, # *2 speeds up training
        use_gradient_checkpointing = "unsloth", # Reduces memory usage
        random_state = 42,
    )



In [20]:
model.generation_config.max_length = None
tokenizer.chat_template = get_chat_template()

#### Define Reward Function


In [21]:
import random
import sys
from pathlib import Path
from datasets import Dataset

base_dir = Path.cwd().parent
print("Notebook cwd:", base_dir)

project_dir = base_dir / "src"
print("Project dir:", project_dir)

if not project_dir.exists():
    raise FileNotFoundError(f"Folder not found: {project_dir}")

# Add the project folder to Python path
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print("Added to sys.path:", sys.path[0])

from minesweeper.engine import GameConfig, GameEngine

#Create the game first
def sample_prefill_move(game: GameEngine, rng: random.Random, reveal_probability=0.7):
    reveal_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]
    flag_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]

    if not reveal_candidates and not flag_candidates:
        return None

    if reveal_candidates and (not flag_candidates or rng.random() < reveal_probability):
        tile = rng.choice(reveal_candidates)
        return {"action": "reveal", "x": tile.x, "y": tile.y}

    tile = rng.choice(flag_candidates)
    return {"action": "flag", "x": tile.x, "y": tile.y}


def create_game(width=9, height=9, mine_density=0.15, seed=None, output_format="compact", max_prefill_moves=4):
    rng = random.Random(seed) if seed is not None else random.Random()
    game = GameEngine(config=GameConfig(width=width, height=height, mine_density=mine_density), rng=rng)

    prefill_moves = rng.randint(0, max_prefill_moves)

    if prefill_moves > 0:
        first_x = rng.randrange(width)
        first_y = rng.randrange(height)
        game.reveal(first_x, first_y)

        for _ in range(prefill_moves - 1):
            if game.status.value != "in_progress":
                break

            move = sample_prefill_move(game, rng)
            if move is None:
                break

            try:
                if move["action"] == "reveal":
                    game.reveal(move["x"], move["y"])
                else:
                    game.flag(move["x"], move["y"])
            except ValueError:
                continue

    visible_state = game.compact_state()
    full_board_state = game.full_board_compact_state() if game.snapshot()["mines_placed"] else None
    return [visible_state, full_board_state, game.snapshot()]

#create a batch of games with random configurations
def create_game_batch(num_games = 5):
    board_size = [(5, 5), (9, 9), (12, 12), (15, 15), (15, 20), (12, 15), (20, 20)]
    mine_density = [0.15, 0.30]
    
    games = []
    for _ in range(num_games):
        width, height = random.choice(board_size)
        density = random.choice(mine_density)
        game = create_game(width=width, height=height, mine_density=density)
        games.append(game)
    
    return games

#Helper function to build user prompt from game state
def build_user_prompt_from_state(game):
    user_prompt = f"Board State: {game['board']}\nMax Mines: {game['mine_count']}\nMax Rows: {game['height']}\nMax Columns: {game['width']}"
    return user_prompt


def create_dataset_from_games(games):
    rows = []
    # Set to None to use raw user prompts without chat template formatting
    for game in games:
        user_prompt = build_user_prompt_from_state(game[0])
        rows.append(
            {
                "prompt": tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": get_system_prompt()},
                        {"role": "user", "content": user_prompt}
                    ],
                    tokenize=False,
                    add_generation_prompt=True
                ) ,
                "revealed_board": str(game[1]),
                "snapshot": str(game[2]),
            }
        )
    return Dataset.from_list(rows)


def build_live_dataset(num_examples, board_sizes, mine_densities, seed=None):
  rng = random.Random(seed)
  rows = []

  while len(rows) < num_examples:
      width, height = rng.choice(board_sizes)
      density = rng.choice(mine_densities)

      game = create_game(width=width, height=height, mine_density=density)
      visible_state, full_state, snapshot = game[0], game[1], game[2]

      if visible_state["status"] != "in_progress":
          continue

      user_prompt = build_user_prompt_from_state(visible_state)

      rows.append({
          "prompt": tokenizer.apply_chat_template(
              [
                  {"role": "system", "content": get_system_prompt()},
                  {"role": "user", "content": user_prompt},
              ],
              tokenize=False,
              add_generation_prompt=True,
          ),
          "rows": visible_state["height"],
          "columns": visible_state["width"],
          "max_mines": visible_state["mine_count"],
          "board_state": str(visible_state["board"]),
          "revealed_board": str(full_state),
          "snapshot": str(snapshot),
      })

  return Dataset.from_list(rows)    # str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")


# game = create_game()
# print(game)  

    # str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")


Notebook cwd: d:\Projects\Minesweeper LLM
Project dir: d:\Projects\Minesweeper LLM\src
Added to sys.path: d:\Projects\Minesweeper LLM\src
pygame 2.6.1 (SDL 2.28.4, Python 3.12.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [22]:
import ast
import json
import re
from minesweeper.rl_rewards import calculate_reward_components

rewards = []

MALFORMED_COMPLETION_PENALTY = -1.0
SNAPSHOT_LOAD_PENALTY = -1.0
INVALID_MOVE_PENALTY = -0.8
EXECUTION_ERROR_PENALTY = -0.66

def build_game_from_state(snapshot):
    game = GameEngine.from_snapshot(ast.literal_eval(snapshot))
    return game

def parse_completion_payload(completion):
    if not isinstance(completion, str):
        raise TypeError("Completion must be a JSON string.")
    payload = json.loads(completion.strip())
    if not isinstance(payload, dict):
        raise TypeError("Completion payload must decode to a dict.")
    nested_action = payload.get("action")
    if isinstance(nested_action, dict):
        if {"action", "x", "y"}.issubset(nested_action):
            payload = nested_action
        elif isinstance(nested_action.get("type"), str):
            payload = dict(payload)
            payload["action"] = nested_action["type"]
    return payload

def validate_move(rows, columns, response):
    if not isinstance(response, dict):
        return False
    action = response.get("action")
    x = response.get("x")
    y = response.get("y")

    if not isinstance(action, str):
        return False
    if action not in {"reveal", "flag"}:
        return False
    if not isinstance(x, int) or not isinstance(y, int):
        return False
    if x < 0 or x >= columns or y < 0 or y >= rows:
        return False
    return True

def get_reward_context(i, rows_list, columns_list, snapshots):
    rows = rows_list[i] if i < len(rows_list) else None
    columns = columns_list[i] if i < len(columns_list) else None
    snapshot = snapshots[i] if i < len(snapshots) else None
    if snapshot is None or rows is None or columns is None:
        raise ValueError(f"Missing required snapshot information for reward calculation at index {i}")
    return rows, columns, snapshot

def score_move_reward(game, response):
    return calculate_reward_components(game, response)

def score_completion_reward(i, completion, rows_list, columns_list, snapshots):
    rows, columns, snapshot = get_reward_context(i, rows_list, columns_list, snapshots)

    try:
        response = parse_completion_payload(completion)
    except Exception:
        return {
            "reward": float(MALFORMED_COMPLETION_PENALTY),
            "component": "malformed_completion",
        }

    try:
        game = build_game_from_state(snapshot)
    except Exception:
        return {
            "reward": float(SNAPSHOT_LOAD_PENALTY),
            "component": "snapshot_load_failure",
        }

    if not validate_move(rows, columns, response):
        return {
            "reward": float(INVALID_MOVE_PENALTY),
            "component": "invalid_move",
        }

    try:
        reward_components = score_move_reward(game, response)
    except ValueError as e:
        error_string = str(e)
        if "Revealed tiles cannot be flagged." in error_string:
            return {
                "reward": float(-0.5),
                "component": "revealed tiles cannot be flagged",
            }
                
        return {
            "reward": float(EXECUTION_ERROR_PENALTY),
            "component": "execution_error",
        }

    return {
        "reward": float(reward_components["total_reward"]),
        "component": "move_reward",
        "reward_components": reward_components,
    }

def calculate_reward(prompts=None, completions=None, **kwargs):
    rows_list = kwargs.get("rows") or []
    columns_list = kwargs.get("columns") or []
    snapshots = kwargs.get("snapshot") or []

    rewards = []
    reward_logs = []

    for i, completion in enumerate(completions or []):
        reward_result = score_completion_reward(i, completion, rows_list, columns_list, snapshots)
        rewards.append(reward_result["reward"])
        reward_logs.append(reward_result)

    calculate_reward.last_logs = reward_logs
    return rewards

#### Set GRPO config and sampling parameters

In [23]:
from trl import GRPOConfig, GRPOTrainer
from minesweeper.wandb_logging import WandbRewardLoggerCallback
from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    top_p=0.95,
    top_k=50,
    seed=42,
    temperature = 1.5,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    use_vllm=False,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_generations=16,
    max_prompt_length=max_seq_length - 32,
    max_completion_length=32,
    max_steps=500,
    save_steps=40,
    report_to="wandb",
    run_name="minesweeper-grpo-debug",
    output_dir="outputs",
    max_grad_norm = 1.0
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [24]:
# GRPO trainer using live engine states instead of the offline CSV dataset
board_sizes = [(5, 5)]
mine_densities = [0.15, 0.30]
live_train_dataset = build_live_dataset(
    num_examples=500,
    board_sizes=board_sizes,
    mine_densities=mine_densities,
    seed=42
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[calculate_reward],
    args=training_args,
    train_dataset=live_train_dataset,
)

grpo_trainer.add_callback(WandbRewardLoggerCallback(calculate_reward))
grpo_trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRE

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / calculate_reward / mean,rewards / calculate_reward / std
1,0.001986,-0.723828,1.418166,19.125000,17.000000,23.000000,0.000000,19.125000,17.000000,23.000000,1.986515,-0.723828,1.418166
2,0.002056,-0.771875,1.383832,19.562500,18.000000,29.000000,0.000000,19.562500,18.000000,29.000000,2.055601,-0.771875,1.383832
3,0.002047,-1.259375,1.284292,20.687500,14.000000,32.000000,0.125000,19.071430,14.000000,27.000000,2.047486,-1.259375,1.284292
4,0.001694,-0.700417,1.394394,19.125000,13.000000,24.000000,0.000000,19.125000,13.000000,24.000000,1.694016,-0.700417,1.394394
5,0.001898,0.164405,0.947622,19.000000,19.000000,19.000000,0.000000,19.000000,19.000000,19.000000,1.898057,0.164405,0.947622
6,0.002056,-0.987500,0.616306,19.937500,17.000000,32.000000,0.062500,19.133335,17.000000,23.000000,2.056142,-0.987500,0.616306
7,0.002425,-0.975000,0.837854,21.375000,17.000000,32.000000,0.125000,19.857143,17.000000,25.000000,2.425128,-0.975000,0.837854
8,0.001714,-0.383750,1.523154,19.062500,19.000000,20.000000,0.000000,19.062500,19.000000,20.000000,1.714059,-0.383750,1.523154
9,0.001926,0.346500,0.438925,19.062500,19.000000,20.000000,0.000000,19.062500,19.000000,20.000000,1.925909,0.346500,0.438925
10,0.001959,0.501000,0.028519,19.000000,19.000000,19.000000,0.000000,19.000000,19.000000,19.000000,1.957506,0.501000,0.028519


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-40\tokenizer_config.json.
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


KeyboardInterrupt: 

Start reinforcement training of the model using GRPO

In [ ]:
grpo_trainer.train()

#### Save the GRPO trained model

In [ ]:
save_model() 
save_model_gguf()

#### Play a full game with the learned model

In [ ]:
import json
import random

from minesweeper.engine import GameConfig, GameEngine
from unsloth import FastLanguageModel


def render_compact_board(board):
    return "\n".join(" ".join(str(cell) for cell in row) for row in board)


def build_live_user_prompt(state):
    return (
        f"Board State: {state['board']}\n"
        f"Score: {state['score']}\n"
        f"Max Mines: {state['mine_count']}\n"
        f"Max Rows: {state['height']}\n"
        f"Max Columns: {state['width']}"
    )


def generate_model_move(state, *, max_new_tokens=32):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_live_user_prompt(state)},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    stop_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if (
        im_end_id is not None
        and im_end_id != tokenizer.unk_token_id
        and im_end_id not in stop_ids
    ):
        stop_ids.append(im_end_id)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=False).strip()

    for stop_token in ("<|im_end|>", "<|endoftext|>"):
        if stop_token in generated_text:
            generated_text = generated_text.split(stop_token, 1)[0].strip()

    move = parse_completion_payload(generated_text)
    return generated_text, move


def play_full_game_with_model(width=5, height=5, mine_density=0.15, seed=42, max_turns=200):
    FastLanguageModel.for_inference(model)
    game = GameEngine(
        config=GameConfig(width=width, height=height, mine_density=mine_density),
        rng=random.Random(seed),
    )

    turn = 1
    while game.status.value == "in_progress" and turn <= max_turns:
        state = game.compact_state()
        print(f"Turn {turn}")
        print(render_compact_board(state["board"]))
        print(f"Score: {state['score']} | Status: {state['status']} | Flags: {state['flagged_count']}/{state['mine_count']}")

        try:
            raw_move, move = generate_model_move(state)
            print(f"Model move: {raw_move}")
            if move["action"] == "reveal":
                result = game.reveal(move["x"], move["y"])
            else:
                result = game.flag(move["x"], move["y"])
            print(f"Result: {result.message} | Move score delta: {result.score_delta:+d}")
        except Exception as exc:
            print(f"Model move failed: {exc}")
            continue

        updated_state = game.compact_state()
        print(render_compact_board(updated_state["board"]))
        print(f"Score: {updated_state['score']} | Status: {updated_state['status']} | Flags: {updated_state['flagged_count']}/{updated_state['mine_count']}")
        print("-" * 60)
        turn += 1

    final_state = game.compact_state()
    print("Final board state:")
    print(render_compact_board(final_state["board"]))
    print(f"Final score: {final_state['score']} | Final status: {final_state['status']}")
    return game


# Example run
played_game = play_full_game_with_model(width=5, height=5, mine_density=0.15, seed=42)


Turn 1
. . . . .
. . . . .
. . . . .
. . . . .
. . . . .
Score: 0 | Status: in_progress | Flags: 0/4
Model move: {"action": "reveal", "x": 2, "y": 2}
Result: Reveal processed. | Move score delta: +30
. . . . .
1 1 1 1 1
0 0 0 0 0
1 1 1 1 1
. . . . .
Score: 30 | Status: in_progress | Flags: 0/4
------------------------------------------------------------
Turn 2
. . . . .
1 1 1 1 1
0 0 0 0 0
1 1 1 1 1
. . . . .
Score: 30 | Status: in_progress | Flags: 0/4
Model move: {"action": "flag", "x": 2, "y": 3}
Model move failed: Revealed tiles cannot be flagged.
Turn 2
. . . . .
1 1 1 1 1
0 0 0 0 0
1 1 1 1 1
. . . . .
Score: 30 | Status: in_progress | Flags: 0/4
Model move: {"action": "flag", "x": 2, "y": 3}
Model move failed: Revealed tiles cannot be flagged.
Turn 2
. . . . .
1 1 1 1 1
0 0 0 0 0
1 1 1 1 1
. . . . .
Score: 30 | Status: in_progress | Flags: 0/4
Model move: {"action": "flag", "x": 2, "y": 3}
Model move failed: Revealed tiles cannot be flagged.
Turn 2
. . . . .
1 1 1 1 1
0 0 0 0 0
1